## Metadata and filtered search with **nemo_retriever**

This notebook mirrors `metadata_and_filtered_search.ipynb`, but uses the **NeMo Retriever** graph pipeline (`GraphIngestor`), **LanceDB** for the vector store, and **`IngestVdbOperator`** sidecar metadata (`meta_dataframe`, `meta_source_field`, `meta_fields`).

User columns are merged into each chunk’s **`content_metadata`**, which is what gets serialized into LanceDB’s `metadata` JSON column. For retrieval, this notebook uses **`Retriever`** from `nemo_retriever` and post-filters hits with **`filter_hits_by_content_metadata`** (LanceDB rows store that JSON as a string; Milvus-style server-side `content_metadata["meta_a"]` filters are not wired here).

Prerequisites: PDFs under `NEMO_RETRIEVER_ROOT/data/` (or edit paths), Python env with `nemo-retriever` and dependencies, and enough resources for extraction + embedding (set `local_ingest_embed_backend="hf"` if you are not running vLLM locally).

In [ ]:
import os
import shutil
from pathlib import Path

import pandas as pd

from nemo_retriever.graph_ingestor import GraphIngestor
from nemo_retriever.params import EmbedParams, ExtractParams
from nemo_retriever.retriever import Retriever
from nemo_retriever.vdb import IngestVdbOperator, filter_hits_by_content_metadata, parse_hit_content_metadata

# Repo root containing sample data (adjust if your tree differs)
NEMO_RETRIEVER_ROOT = Path(os.environ.get("NEMO_RETRIEVER_ROOT", "/raid/nv-ingest")).resolve()

### Configuration

Use the same embedding model family as ingestion for queries. Point `LANCEDB_URI` at a fresh directory for each full re-ingest (`overwrite=True` by default).

In [ ]:
model_name = "nvidia/llama-nemotron-embed-1b-v2"
LANCEDB_URI = str(Path("./nemo_retriever_meta_lancedb").resolve())
TABLE_NAME = "nv-ingest"

pdf_a = NEMO_RETRIEVER_ROOT / "data" / "woods_frost.pdf"
pdf_b = NEMO_RETRIEVER_ROOT / "data" / "multimodal_test.pdf"
files = [str(p) for p in (pdf_a, pdf_b) if p.is_file()]
if len(files) < 2:
    raise FileNotFoundError(
        f"Expected sample PDFs at {pdf_a} and {pdf_b}. Set NEMO_RETRIEVER_ROOT or copy data files."
    )

### Sidecar metadata file

Supported file types when passing a path: **csv**, **json**, **parquet** (loaded via pandas). The join column (`source` here) must match the **absolute** document path stored on each row (`meta_join_key="auto"` tries `source_id` then `source_name`).

In [ ]:
meta_df = pd.DataFrame(
    {
        "source": [str(pdf_a.resolve()), str(pdf_b.resolve())],
        "meta_a": ["alpha", "bravo"],
        "meta_b": [5, 10],
        "meta_c": [True, False],
        "meta_d": [10.0, 20.0],
    }
)
meta_path = Path("./nemo_retriever_meta_sidecar.csv").resolve()
meta_df.to_csv(meta_path, index=False)
meta_path

### 1) Graph ingest (extract + embed)

`inprocess` avoids Ray for a single-machine notebook. Switch to `batch` for large corpora. Use **`local_ingest_embed_backend="hf"`** when you do not have a local vLLM embed server.

In [ ]:
extract_params = ExtractParams(
    extract_text=True,
    extract_tables=True,
    extract_charts=True,
    extract_images=True,
)
embed_params = EmbedParams(
    model_name=model_name,
    embed_granularity="page",
    local_ingest_embed_backend="hf",
)

ingestor = (
    GraphIngestor(run_mode="inprocess", documents=files)
    .extract(extract_params)
    .embed(embed_params)
)
result_df = ingestor.ingest()
records = result_df.to_dict("records")
len(records)

### 2) Upload to LanceDB with sidecar metadata

`IngestVdbOperator` converts graph rows to NV-Ingest records, merges `meta_fields` into **`content_metadata`**, then calls `LanceDB.run`.

In [ ]:
if Path(LANCEDB_URI).exists():
    shutil.rmtree(LANCEDB_URI)

vdb_kwargs = {
    "uri": LANCEDB_URI,
    "table_name": TABLE_NAME,
    "overwrite": True,
    "meta_dataframe": str(meta_path),
    "meta_source_field": "source",
    "meta_fields": ["meta_a", "meta_b", "meta_c", "meta_d"],
    "meta_join_key": "auto",
}

uploader = IngestVdbOperator(vdb_op="lancedb", vdb_kwargs=vdb_kwargs)
uploader.process(records)

### 3) Retrieval + metadata filter

Run vector search, then keep hits whose parsed `content_metadata` satisfies your predicate (analogous to `content_metadata["meta_a"] == "alpha"`).

In [ ]:
retriever = Retriever(
    vdb="lancedb",
    vdb_kwargs={"uri": LANCEDB_URI, "table_name": TABLE_NAME},
    embedder=model_name,
    top_k=20,
    local_query_embed_backend="hf",
)

queries = ["this is expensive"]
for que in queries:
    hits = retriever.query(que, top_k=20)
    only_alpha = filter_hits_by_content_metadata(hits, lambda m: m.get("meta_a") == "alpha")
    print("raw hits:", len(hits), "filtered (meta_a==alpha):", len(only_alpha))
    for h in only_alpha[:3]:
        print(parse_hit_content_metadata(h).get("meta_a"), h.get("text", "")[:120])

In [ ]:
for que in queries:
    hits = retriever.query(que, top_k=20)
    ge5 = filter_hits_by_content_metadata(hits, lambda m: m.get("meta_b") is not None and m["meta_b"] >= 5)
    print("meta_b >= 5:", len(ge5))

In [ ]:
for que in queries:
    hits = retriever.query(que, top_k=20)
    mc = filter_hits_by_content_metadata(hits, lambda m: m.get("meta_c") is True)
    print("meta_c is True:", len(mc))

In [ ]:
for que in queries:
    hits = retriever.query(que, top_k=20)
    md = filter_hits_by_content_metadata(hits, lambda m: m.get("meta_d") is not None and float(m["meta_d"]) < 20)
    print("meta_d < 20:", len(md))

### CLI equivalent

You can pass the same sidecar options to **`retriever pipeline run`**:

```bash
retriever pipeline run /path/to/pdfs --run-mode inprocess \
  --vdb-kwargs-json '{"uri":"./lancedb","table_name":"nv-ingest"}' \
  --meta-dataframe ./nemo_retriever_meta_sidecar.csv \
  --meta-source-field source \
  --meta-fields meta_a,meta_b,meta_c,meta_d
```

Or include `meta_dataframe`, `meta_source_field`, `meta_fields`, and optional `meta_join_key` inside `--vdb-kwargs-json`.